# NBA API — Initial EDA

Exploring the free [`nba_api`](https://github.com/swar/nba_api) (no key required) to understand its structure and what data we can pull.

We'll use the `LeagueDashPlayerStats` endpoint — season-level totals for every player, the same source the statlas warehouse ingests from.

In [1]:
import nba_api
import pandas as pd
from nba_api.stats.endpoints import leaguedashplayerstats

print("nba_api version:", nba_api.__version__)
print("pandas version :", pd.__version__)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

nba_api version: 1.11.4
pandas version : 3.0.3


## Pull a season of player stats

One call returns a `pandas.DataFrame` of season totals for every player who logged minutes.

In [2]:
SEASON = "2024-25"

resp = leaguedashplayerstats.LeagueDashPlayerStats(season=SEASON)
df = resp.get_data_frames()[0]

print(f"Season {SEASON}: {df.shape[0]} players x {df.shape[1]} columns")

Season 2024-25: 569 players x 67 columns


## Basic info

Shape, column list, dtypes, and a peek at the data.

In [3]:
# All available columns
print(f"{len(df.columns)} columns:\n")
print(list(df.columns))

67 columns:

['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK', 'TD3_RANK', 'WNBA_FANTASY_PTS_RANK', 'TEAM_COUNT']


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 67 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   PLAYER_ID              569 non-null    int64  
 1   PLAYER_NAME            569 non-null    str    
 2   NICKNAME               569 non-null    str    
 3   TEAM_ID                569 non-null    int64  
 4   TEAM_ABBREVIATION      569 non-null    str    
 5   AGE                    569 non-null    float64
 6   GP                     569 non-null    int64  
 7   W                      569 non-null    int64  
 8   L                      569 non-null    int64  
 9   W_PCT                  569 non-null    float64
 10  MIN                    569 non-null    float64
 11  FGM                    569 non-null    int64  
 12  FGA                    569 non-null    int64  
 13  FG_PCT                 569 non-null    float64
 14  FG3M                   569 non-null    int64  
 15  FG3A             

In [5]:
# A readable slice of the most common box-score columns
cols = ["PLAYER_NAME", "TEAM_ABBREVIATION", "AGE", "GP", "MIN",
        "PTS", "REB", "AST", "STL", "BLK", "FG_PCT", "FG3_PCT"]
df[cols].head(10)

,PLAYER_NAME,TEAM_ABBREVIATION,AGE,GP,MIN,PTS,REB,AST,STL,BLK,FG_PCT,FG3_PCT
0,A.J. Lawson,TOR,24.0,26,486.410000,236,86,31,13,6,0.421,0.327
1,AJ Green,MIL,25.0,73,1659.141667,541,174,108,37,7,0.429,0.427
2,AJ Johnson,WAS,20.0,29,638.386667,220,59,76,12,3,0.385,0.267
3,Aaron Gordon,DEN,29.0,51,1446.716667,748,247,164,23,14,0.531,0.436
4,Aaron Holiday,HOU,28.0,62,792.246667,340,78,83,19,11,0.437,0.398
5,Aaron Nesmith,IND,25.0,45,1121.618333,541,178,54,35,17,0.507,0.431
6,Aaron Wiggins,OKC,26.0,76,1743.900000,914,295,134,60,18,0.488,0.383
7,Adam Flagler,OKC,25.0,37,202.965000,65,27,12,8,3,0.260,0.194
8,Adama Sanogo,CHI,23.0,4,21.498333,8,6,1,0,0,0.571,0.000
9,Adem Bona,PHI,22.0,58,904.550000,337,245,27,26,69,0.703,0.000


In [6]:
# Numeric summary
df[["AGE", "GP", "MIN", "PTS", "REB", "AST"]].describe()

,AGE,GP,MIN,PTS,REB,AST
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,26.230228,46.231986,1043.233743,492.108963,190.713533,114.783831
std,4.220348,24.357135,805.522063,473.842651,176.520449,129.328428
min,19.000000,1.000000,2.601667,0.000000,0.000000,0.000000
25%,23.000000,25.000000,314.276667,107.000000,50.000000,22.000000
50%,25.000000,50.000000,907.458333,356.000000,158.000000,75.000000
75%,29.000000,68.000000,1716.348333,732.000000,273.000000,156.000000
max,40.000000,82.000000,3036.076667,2484.000000,1010.000000,880.000000


## Quick sanity check — top scorers

Totals, so sort by `PTS`. (For per-game or leaderboards we'd apply a minimum-games qualifier — see the warehouse reliability rule.)

In [7]:
top10 = df.sort_values("PTS", ascending=False).head(10)
top10[["PLAYER_NAME", "TEAM_ABBREVIATION", "GP", "PTS"]].reset_index(drop=True)

,PLAYER_NAME,TEAM_ABBREVIATION,GP,PTS
0,Shai Gilgeous-Alexander,OKC,76,2484
1,Anthony Edwards,MIN,79,2177
2,Nikola Jokić,DEN,70,2071
3,Giannis Antetokounmpo,MIL,67,2036
4,Jayson Tatum,BOS,72,1932
5,Devin Booker,PHX,75,1923
6,Trae Young,ATL,76,1841
7,Tyler Herro,MIA,77,1840
8,Cade Cunningham,DET,70,1830
9,James Harden,LAC,79,1802
